# 네이버 영화리뷰 감성 분석

### 01. 데이터 로드

**✅ 실습 과제**
- [네이버 영화 리뷰 데이터셋](https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt)을 불러온다.
- 데이터의 컬럼 구성과 샘플을 확인한다.

**🔍 확인 질문**
1. 리뷰 텍스트와 정답 라벨은 각각 어떤 컬럼에 저장되어 있는가?
2. 긍정 / 부정 라벨은 어떤 값으로 표현되어 있는가?

In [56]:
# ! pip install tensorflow

In [57]:
# 데이터 다운로드
from tensorflow.keras.utils import get_file

ratings_train_path = get_file("ratings_train.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")
ratings_test_path = get_file("ratings_test.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt")

ratings_train_path, ratings_test_path

('C:\\Users\\Playdata\\.keras\\datasets\\ratings_train.txt',
 'C:\\Users\\Playdata\\.keras\\datasets\\ratings_test.txt')

In [58]:
# 데이터프레임 생성
import pandas as pd

train_df = pd.read_table(ratings_train_path)
test_df = pd.read_table(ratings_test_path)

print(train_df.head())
print(train_df.columns)
print(train_df.shape)


         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
Index(['id', 'document', 'label'], dtype='str')
(150000, 3)


In [59]:
# 데이터 샘플링
train_df = train_df.sample(5000, random_state=42)
test_df = test_df.sample(1000, random_state=42)

print(train_df.shape)
print(test_df.shape)

(5000, 3)
(1000, 3)


### 02. 데이터 전처리

##### 02-01. 한글 전처리

**✅ 실습 과제**
- 특수문자, 숫자, 불필요한 기호를 제거한다.
- 정규표현식을 사용하여 한글만 남긴다.

**🔍 확인 질문**
1. 한글 전처리를 하지 않으면 어떤 문제가 발생할 수 있는가?
2. 감성 분석에서 이모지나 느낌표는 제거하는 것이 항상 옳은가?




In [60]:
# 결측치 제거
import re

def clean_korean(text):
    
    if isinstance(text, str):
        text = re.sub(r"[^가-힣\s]", "", text)   # 한글과 공백만 남김
        text = re.sub(r"\s+", " ", text)        # 공백 정리
        return text.strip()
    
    return ""

train_df["clean_text"] = train_df["document"].apply(clean_korean)
test_df["clean_text"] = test_df["document"].apply(clean_korean)

train_df[["document","clean_text"]].head()

,document,clean_text
59770,수OO만에 다시보네여,수만에 다시보네여
21362,일방적인 영화다. 관객 좀 고려해주시길,일방적인 영화다 관객 좀 고려해주시길
127324,세상을 초월하는 한 사람의 선한 마음,세상을 초월하는 한 사람의 선한 마음
140509,멍하다.. 여러생각이 겹치는데 오랜만에 영화 보고 이런 느낌 느껴본다,멍하다 여러생각이 겹치는데 오랜만에 영화 보고 이런 느낌 느껴본다
144297,"우와 별 반개도 아까운판에 밑에 CJ 알바생들 쩐다.. 전부 만점이야 ㅎㅎㅎ..,....",우와 별 반개도 아까운판에 밑에 알바생들 쩐다 전부 만점이야 야 그만해라 저영화는 ...


In [61]:
# 한글 토큰화 전처리 (특수문자 처리, 어간 추출, 불용어 처리) -> 함수
stop_words = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']
from konlpy.tag import Okt
import re

okt = Okt()

stop_words = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

def preprocess_korean(text):

    # 1. 한글만 남기기
    text = re.sub(r"[^가-힣\s]", "", str(text))

    # 2. 형태소 분석 + 어간 추출
    tokens = okt.pos(text, stem=True)

    # 3. 불용어 제거 + 품사 필터링
    tokens = [
        word for word, pos in tokens
        if pos in ["Noun","Verb","Adjective"]
        and word not in stop_words
        and len(word) > 1
    ]

    return tokens

train_df["tokens"] = train_df["clean_text"].apply(preprocess_korean)
test_df["tokens"] = test_df["clean_text"].apply(preprocess_korean)

train_df[["clean_text","tokens"]].head()

,clean_text,tokens
59770,수만에 다시보네여,"[수만, 다시, 보다]"
21362,일방적인 영화다 관객 좀 고려해주시길,"[일방, 영화, 관객, 고려, 해주다]"
127324,세상을 초월하는 한 사람의 선한 마음,"[세상, 초월, 사람, 선하다, 마음]"
140509,멍하다 여러생각이 겹치는데 오랜만에 영화 보고 이런 느낌 느껴본다,"[멍하다, 생각, 치다, 영화, 보고, 이렇다, 느낌, 느끼다, 보다]"
144297,우와 별 반개도 아까운판에 밑에 알바생들 쩐다 전부 만점이야 야 그만해라 저영화는 ...,"[반개, 아깝다, 알바생, 쩐다, 전부, 만점, 그만하다, 영화, 정말, 쓰레기, ..."


##### 02-02. Tokenizing & Sequencing

**✅ 실습 과제**
- Tokenizer를 사용해 단어 사전을 생성한다.
- 문장을 정수 시퀀스로 변환한다.
- padding을 적용하여 시퀀스 길이를 맞춘다.

**🔍 확인 질문**
1. `num_words` 파라미터는 어떤 역할을 하는가?
2. padding을 하지 않으면 배치 학습에서 어떤 문제가 발생하는가?



In [62]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 토큰을 다시 문장 형태로 변환
texts = train_df["tokens"].apply(lambda x: " ".join(x)).tolist()

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

print(sequences[:3])

max_len = 30

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post"
)

print(padded_sequences[:3])


[[3125, 37, 2], [2026, 1, 236, 3126, 94], [224, 2027, 21, 3127, 96]]
[[3125   37    2    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [2026    1  236 3126   94    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 224 2027   21 3127   96    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]]


In [63]:
import torch
import torchtext
print(torch.__version__)
print(torchtext.__version__)

# sequence 작업 (단어사전 생성, 텍스트-수열 변환)
from torchtext.vocab import build_vocab_from_iterator

# 값을 하나씩 반환하는 함수
def yield_tokens(data):
    for tokens in data:
        yield tokens

2.2.2+cpu
0.17.2+cpu


In [64]:
vocab = build_vocab_from_iterator(
    yield_tokens(train_df["tokens"]),
    specials=["<unk>", "<pad>"]
)

In [65]:
# padding 작업
sequences = [vocab(tokens) for tokens in train_df["tokens"]]

In [66]:
from torch.nn.utils.rnn import pad_sequence
import torch

# tensor 변환
sequences = [torch.tensor(seq) for seq in sequences]

# padding
padded_sequences = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=vocab["<pad>"]
)

print(padded_sequences.shape)

torch.Size([5000, 37])


In [67]:
max_len = 30

sequences = [torch.tensor(seq[:max_len]) for seq in sequences]

padded_sequences = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=vocab["<pad>"]
)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_17548\3388586674.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequences = [torch.tensor(seq[:max_len]) for seq in sequences]


##### 02-03. Sequence Decoding

**✅ 실습 과제**
- 정수 시퀀스를 다시 텍스트로 복원해본다.
- 토큰 인덱스와 단어의 매핑 관계를 확인한다.

**🔍 확인 질문**
1. `<OOV>` 토큰은 언제 사용되는가?
2. 디코딩 결과가 원문과 완전히 동일하지 않은 이유는 무엇인가?


In [68]:
itos = vocab.get_itos()

print(itos[:20])

['<unk>', '<pad>', '영화', '보다', '없다', '있다', '좋다', '정말', '재밌다', '되다', '진짜', '이다', '같다', '아니다', '나오다', '않다', '평점', '만들다', '연기', '최고']


In [69]:
sequence = sequences[0]

decoded_sentence = [itos[idx] for idx in sequence]

print(decoded_sentence)

['수만', '다시', '보다']


In [70]:
decoded_text = " ".join(decoded_sentence)
print(decoded_text)

수만 다시 보다


In [71]:
print("원문 :", train_df["clean_text"].iloc[0])
print("디코딩 :", decoded_text)

원문 : 수만에 다시보네여
디코딩 : 수만 다시 보다


### 03. 모델 생성 및 학습

**✅ 실습 과제**
- Embedding Layer를 포함한 감성분석 모델을 정의한다.
- 손실 함수와 옵티마이저를 설정한다.
- 학습 과정을 통해 loss 변화를 확인한다.

**🔍 확인 질문**
1. Embedding Layer는 어떤 역할을 하는가?
2. One-hot 인코딩 대신 Embedding을 사용하는 이유는 무엇인가?

In [72]:
import torch
import torch.nn as nn

class SentimentModel(nn.Module):
    
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        
        x = self.embedding(x)          # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)              # 문장 벡터 평균
        x = self.fc(x)
        x = self.sigmoid(x)

        return x

In [73]:

vocab_size = len(vocab)
embed_dim = 100

model = SentimentModel(vocab_size, embed_dim)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [74]:
import torch

labels = torch.tensor(train_df["label"].values, dtype=torch.float32)

In [75]:
epochs = 5

X = padded_sequences
y = labels.float()

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X).squeeze()

    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    epochs = 5

X = padded_sequences
y = labels

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X).squeeze()

    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6949
Epoch 2, Loss: 0.6933
Epoch 3, Loss: 0.6931
Epoch 4, Loss: 0.6932
Epoch 5, Loss: 0.6929
Epoch 1, Loss: 0.6924
Epoch 2, Loss: 0.6918
Epoch 3, Loss: 0.6914
Epoch 4, Loss: 0.6912
Epoch 5, Loss: 0.6911


In [76]:
# 모델 정의

import torch
import torch.nn as nn

class SentimentModel(nn.Module):
    
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        
        # Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 출력 레이어
        self.fc = nn.Linear(embed_dim, 1)
        
        # 활성화 함수
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        
        x = self.embedding(x)        # (batch, seq_len, embed_dim)
        x = x.mean(dim=1)            # 문장 벡터 평균
        x = self.fc(x)
        x = self.sigmoid(x)

        return x
    
    vocab_size = len(vocab)
embed_dim = 100

model = SentimentModel(vocab_size, embed_dim)

print(model)

SentimentModel(
  (embedding): Embedding(7259, 100)
  (fc): Linear(in_features=100, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [77]:
# 모델 인스턴스 생성

vocab_size = len(vocab)
embed_dim = 100

model = SentimentModel(vocab_size, embed_dim)

In [78]:
# 학습

epochs = 5

X = padded_sequences
y = labels

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X).squeeze()

    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7527
Epoch 2, Loss: 0.7527
Epoch 3, Loss: 0.7527
Epoch 4, Loss: 0.7527
Epoch 5, Loss: 0.7527


### 04. 모델 평가

**✅ 실습 과제**
- 검증 데이터로 모델 성능을 평가한다.
- 정확도(acc)와 손실(loss)을 확인한다.

**🔍 확인 질문**
1. 학습 데이터 성능과 평가 데이터 성능 차이가 의미하는 것은 무엇인가?
2. 과적합 여부는 어떻게 판단할 수 있는가?

In [79]:
for tokens in test_df["tokens"]:
    if not isinstance(tokens, list):
        print("Not list:", tokens)
        break

    for t in tokens:
        if not isinstance(t, str):
            print("Not string:", tokens)
            break

In [80]:
print(test_df["tokens"].iloc[0])
print(type(test_df["tokens"].iloc[0]))

['모든', '편견', '날다', '버리다', '가슴', '따뜻하다', '영화', '로버트', '필립', '세이모어', '호프만', '영원하다']
<class 'list'>


In [81]:
test_sequences = []

for tokens in test_df["tokens"]:
    
    if not isinstance(tokens, list):
        continue
    
    # 문자열만 남기기
    clean_tokens = [str(t) for t in tokens if isinstance(t, str) and len(t) > 0]

    if len(clean_tokens) == 0:
        continue

    try:
        seq = vocab.lookup_indices(clean_tokens)
        test_sequences.append(seq)
    except:
        continue

In [82]:
test_sequences = []
test_labels = []

for tokens, label in zip(test_df["tokens"], test_df["label"]):

    if not isinstance(tokens, list):
        continue

    clean_tokens = []

    for t in tokens:
        if isinstance(t, str):
            t = t.strip()
            if t != "":
                clean_tokens.append(t)

    if len(clean_tokens) == 0:
        continue

    try:
        seq = vocab.lookup_indices(clean_tokens)
        test_sequences.append(seq)
        test_labels.append(label)
    except Exception:
        continue


In [83]:
import torch
from torch.nn.utils.rnn import pad_sequence

test_sequences = [torch.tensor(seq) for seq in test_sequences]

X_test = pad_sequence(
    test_sequences,
    batch_first=True,
    padding_value=vocab["<pad>"]
)

y_test = torch.tensor(test_labels, dtype=torch.float32)

In [84]:
model.eval()

with torch.no_grad():

    outputs = model(X_test).squeeze()

    loss = criterion(outputs, y_test)

    preds = (outputs > 0.5).float()

    acc = (preds == y_test).float().mean()

print("Test Loss:", loss.item())
print("Test Accuracy:", acc.item())

Test Loss: 0.7838095426559448
Test Accuracy: 0.46415093541145325


### 05. 모델 추론

**✅ 실습 과제**
- 임의의 문장을 입력하여 감성을 예측한다.
- 출력 확률을 기반으로 긍/부정을 해석한다.

**🔍 확인 질문**
1. 모델 출력값은 확률인가 점수인가?
2. 기준값(threshold)은 왜 필요한가?

In [85]:
import torch
import numpy as np

test_text = ["이 영화 정말 재미있다"]

seq = tokenizer.texts_to_sequences(test_text)

padded = pad_sequences(seq, maxlen=max_len)

input_tensor = torch.tensor(padded, dtype=torch.long)

model.eval()

with torch.no_grad():
    output = model(input_tensor)
    prob = torch.sigmoid(output)

print("예측 확률:", prob.item())

if prob.item() > 0.5:
    print("긍정 리뷰")
else:
    print("부정 리뷰")

예측 확률: 0.5620766878128052
긍정 리뷰
